# SRI research experiments: A100 Colab

This notebook runs **classical**, exploratory experiments: synthetic pre/post-cancellation Xi and MSE, two V3 denominator variants, synchronized GPU scaling, simulated finite precision and phase/gain mismatch, an eager-versus-compiled kernel comparison, and factual multiple-choice comparisons with and without Wikipedia retrieval. It compares confidence-only, Xi-only, and combined warnings using thresholds set on a separate calibration split. A warning causes an abstention in this test, not factual correction.

Run all cells in order. Select **Runtime → Change runtime type → A100 GPU**. Model and dataset downloads and Wikipedia requests require internet. Only public benchmark questions are sent to Wikipedia; uploaded patent files are not used. Retrieval is not checked for factual support, and errors are recorded rather than treated as evidence. The architecture sensitivity test is a simplified digital model, not an actual IC or circuit simulation. There is no physical phase-conjugate hardware, quantum advantage, measured power or proof of hallucination prevention in this notebook.

Results and plots are written under `/content` by default, which is **temporary**. Download the ZIP using the final cell, or enable Google Drive in the configuration cell before the run. Compute-unit usage and runtime depend on Colab, availability, model downloads, GPU and network; no amount is guaranteed.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.45,<5", "datasets>=3,<5", "accelerate>=1,<2",
                "matplotlib", "requests"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Enable a GPU runtime and rerun.")
print("GPU:", torch.cuda.get_device_name(0))
if "A100" not in torch.cuda.get_device_name(0).upper():
    print("WARNING: This is not an A100; results will record the actual GPU.")


## Configure the run

Defaults use 40 calibration questions and 100 held-out questions, plus 180 synthetic numerical trials. The 1.5B model is a baseline, not evidence that larger models behave similarly. Increase these only after a short pilot. Set `SAVE_TO_DRIVE = True` to authorize Google Drive mounting; this notebook will create a fresh time-stamped folder. No repository is changed.

To recover an earlier partial run without repeating completed experiments, set `RESUME_RETRIEVAL = True` and upload its latest recovery ZIP or `sri_results.json` when prompted. Keep the original ZIP: the recovery ZIP contains updated JSON and the factual plot, not the original plots. Requests are paced (at least 10 seconds apart) and HTTP 429 is retried using the server's `Retry-After` and bounded backoff. If rate limiting persists the notebook stops and saves progress; wait before resuming with the newest ZIP. This may take an hour or more. The recorded model revision is used; the old tokenizer revision was not recorded.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import shutil

SAVE_TO_DRIVE = False
RESUME_RETRIEVAL = True  # Set False for a completely new experiment.
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
N_CALIBRATION = 40
N_TEST = 100
N_TRIALS = 20  # per noise type and level
SEED = 108

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive/SRI_Experiments")
else:
    root = Path("/content/SRI_Experiments")
run_dir = root / ("run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
run_dir.mkdir(parents=True, exist_ok=False)
print("Results folder:", run_dir)
source_report = None
if RESUME_RETRIEVAL:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one original experiment ZIP or sri_results.json.")
    name, data = next(iter(uploaded.items()))
    if name.lower().endswith(".zip"):
        import io
        with zipfile.ZipFile(io.BytesIO(data)) as archive:
            matches = [item for item in archive.infolist()
                       if item.filename.rsplit("/", 1)[-1] == "sri_results.json"]
            if len(matches) != 1 or matches[0].file_size > 50_000_000:
                raise ValueError("ZIP must contain exactly one reasonably sized sri_results.json.")
            data = archive.read(matches[0])
    elif not name.lower().endswith(".json"):
        raise ValueError("Upload a ZIP or JSON file.")
    source_report = run_dir / "original_results.json"
    source_report.write_bytes(data)


## Write the self-contained experiment script

The script is included below for transparency and is copied into the results folder for provenance.


In [ ]:
%%writefile /content/sri_colab_experiments.py
"""Colab research harness. See the accompanying notebook for setup and execution."""

import argparse
from email.utils import parsedate_to_datetime
import hashlib
import json
import math
import os
import platform
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from datasets import load_dataset
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from transformers import AutoModelForCausalLM, AutoTokenizer


MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "truthfulqa/truthful_qa"
USER_AGENT = "SRIResearchColab/1.0 (https://github.com/Ceraph-Corporation/sri-quantum; public benchmark research)"


class PacedSession(requests.Session):
    def __init__(self, interval=10.0, max_retries=3, max_wait=300.0):
        super().__init__()
        self.interval = interval
        self.max_retries = max_retries
        self.max_wait = max_wait
        self.next_request = 0.0
        self.rate_limited = False

    def get(self, *args, **kwargs):
        if self.rate_limited:
            raise RuntimeError("Retrieval stopped after HTTP 429; retry later.")
        for attempt in range(self.max_retries + 1):
            wait = self.next_request - time.monotonic()
            if wait > 0:
                time.sleep(wait)
            try:
                response = super().get(*args, **kwargs)
            finally:
                self.next_request = time.monotonic() + self.interval
            if response.status_code != 429:
                return response
            header = response.headers.get("Retry-After")
            delay = 0.0
            if header:
                try:
                    delay = max(0.0, float(header))
                except ValueError:
                    try:
                        date = parsedate_to_datetime(header)
                        if date.tzinfo is None:
                            date = date.replace(tzinfo=timezone.utc)
                        delay = max(0.0, (date - datetime.now(timezone.utc)).total_seconds())
                    except (TypeError, ValueError, OverflowError):
                        print(f"Invalid Retry-After header {header!r}; using backoff.", flush=True)
            delay = max(delay, self.interval * (2 ** (attempt + 1)))
            if attempt == self.max_retries or delay > self.max_wait:
                self.rate_limited = True
                raise RuntimeError(f"Wikipedia HTTP 429 after {attempt + 1} attempt(s); "
                                   f"Retry-After: {header or 'not supplied'}. Wait before resuming.")
            print(f"Wikipedia HTTP 429; waiting {delay:.1f} seconds before retry {attempt + 1}/{self.max_retries}.", flush=True)
            self.next_request = time.monotonic() + delay
        raise RuntimeError("Unexpected retrieval retry state")


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def save_json(path, data):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(json_safe(data), indent=2, allow_nan=False))
    temporary.replace(path)


def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def xi(x):
    """White-paper version: mean absolute value in the denominator."""
    x = x.float()
    return 1.0 - x.var(unbiased=False) / (x.abs().mean() + 1e-7)


def v3(xi_value, gradient_variance, sigma_sq, variant):
    """Both source denominators are retained as explicit alternative hypotheses."""
    denominator = sigma_sq + 1e-7 if variant == "additive" else sigma_sq * 1e-7
    z = 2.0 * (gradient_variance / denominator - 1.0)
    sigmoid = 1.0 / (1.0 + math.exp(-max(-60.0, min(60.0, z))))
    circuit_a = 0.1 * math.exp(max(-80.0, min(80.0, -(1.0 - xi_value))))
    return circuit_a + 50.0 * sigmoid


def smooth(y, width, method):
    padded = F.pad(y[None, None, :], (width // 2, width // 2), mode="replicate")
    windows = padded.unfold(-1, width, 1)
    if method == "mean":
        return windows.mean(dim=-1).flatten()
    return windows.median(dim=-1).values.flatten()


def make_signal(rng, n, noise_std, kind, device):
    t = torch.arange(n, device=device, dtype=torch.float32)
    # Local structure makes a finite-window estimator meaningful; narrow features
    # and steps ensure that smoothing can also destroy signal.
    clean = (torch.sin(t * (2 * math.pi / 67)) +
             0.35 * torch.sin(t * (2 * math.pi / 19)) +
             0.5 * (t > n * 0.57).float())
    clean += 0.8 * torch.exp(-((t - n * 0.35) / 2.0) ** 2)
    noise = torch.randn(n, generator=rng, device=device) * noise_std
    if kind == "impulsive":
        spikes = torch.rand(n, generator=rng, device=device) < 0.03
        noise = noise + spikes * torch.randn(n, generator=rng, device=device) * noise_std * 6
    elif kind == "drift":
        noise = noise + noise_std * 0.7 * (t / n - 0.5)
    return clean, clean + noise


def hist3d(path, groups, title, xlabel, bins=32):
    fig = plt.figure(figsize=(11, 7))
    ax = fig.add_subplot(111, projection="3d")
    data = [np.asarray(values, dtype=float) for _, values in groups]
    finite = [a[np.isfinite(a)] for a in data]
    nonempty = [a for a in finite if len(a)]
    if not nonempty:
        raise ValueError(f"No finite observations available for {path.name}")
    combined = np.concatenate(nonempty)
    low, high = float(combined.min()), float(combined.max())
    if low == high:
        low -= 0.5
        high += 0.5
    edges = np.linspace(low, high, bins + 1)
    colors = plt.get_cmap("tab10")
    from matplotlib.patches import Patch
    for i, ((label, _), vals) in enumerate(zip(groups, finite)):
        counts, _ = np.histogram(vals, bins=edges)
        ax.bar3d(edges[:-1], np.full(bins, i - 0.32), np.zeros(bins),
                 np.diff(edges) * 0.85, 0.64, counts,
                 color=colors(i % 10), alpha=0.8, shade=True)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Condition")
    ax.set_zlabel("Count")
    ax.set_yticks(range(len(groups)), [label for label, _ in groups])
    ax.set_title(title)
    ax.legend(handles=[Patch(color=colors(i % 10), label=label)
                       for i, (label, _) in enumerate(groups)], loc="upper right")
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)


def numerical_study(output, device, seed, trials):
    generator = torch.Generator(device=device).manual_seed(seed)
    records, residual_groups, xi_groups = [], {}, {"Before": [], "After estimated cancellation": []}
    for noise_type in ("gaussian", "impulsive", "drift"):
        for std in (0.25, 0.75, 1.5):
            for trial in range(trials):
                clean, observed = make_signal(generator, 1024, std, noise_type, device)
                estimate = smooth(observed, 5, "median")
                mean_estimate = smooth(observed, 5, "mean")
                estimated_noise = observed - estimate
                paired = 0.5 * (observed + (observed - 2 * estimated_noise))
                gradient_variance = float(torch.diff(observed).var(unbiased=False).item())
                before, after = float(xi(observed).item()), float(xi(paired).item())
                rec = {"noise_type": noise_type, "noise_std": std, "trial": trial,
                       "mse_raw": float(((observed-clean)**2).mean().item()),
                       "mse_mean_filter": float(((mean_estimate-clean)**2).mean().item()),
                       "mse_median_filter": float(((estimate-clean)**2).mean().item()),
                       "mse_paired_estimate": float(((paired-clean)**2).mean().item()),
                       "paired_vs_median_max_abs": float((paired-estimate).abs().max().item()),
                       "xi_before": before, "xi_after": after,
                       "xi_after_constant_shift": float(xi(observed + 10.0).item()),
                       "gradient_variance": gradient_variance, "sigma_sq": 1.0,
                       "chi_additive": v3(before, gradient_variance, 1.0, "additive"),
                       "chi_multiplicative": v3(before, gradient_variance, 1.0, "multiplicative")}
                for variant in ("additive", "multiplicative"):
                    damped = observed * math.exp(-rec["chi_" + variant])
                    rec["mse_damped_" + variant] = float(((damped-clean)**2).mean().item())
                    rec["damped_variance_" + variant] = float(damped.var(unbiased=False).item())
                records.append(rec)
                xi_groups["Before"].append(before)
                xi_groups["After estimated cancellation"].append(after)
                if std == 0.75:
                    residual_groups.setdefault(noise_type + " raw", []).extend(
                        (observed-clean).detach().cpu().tolist()[::4])
                    residual_groups.setdefault(noise_type + " paired", []).extend(
                        (paired-clean).detach().cpu().tolist()[::4])
    hist3d(output / "xi_before_after_3d.png", list(xi_groups.items()),
           "Xi before and after estimated cancellation (all numerical trials)",
           "Xi = 1 - variance / (mean absolute value + epsilon)")
    hist3d(output / "residuals_3d.png", list(residual_groups.items()),
           "Synthetic residuals by noise type, noise std = 0.75",
           "Output minus clean signal")
    hist3d(output / "v3_damping_3d.png",
           [("Additive denominator", [r["chi_additive"] for r in records]),
            ("Multiplicative denominator", [r["chi_multiplicative"] for r in records])],
           "V3 damping by proposed denominator, all numerical trials", "Damping coefficient chi")
    groups = []
    for kind in ("gaussian", "impulsive", "drift"):
        selected = [r for r in records if r["noise_type"] == kind]
        groups.append({"noise_type": kind, "n": len(selected),
                       **{k: float(np.mean([r[k] for r in selected]))
                          for k in ("mse_raw", "mse_mean_filter", "mse_median_filter",
                                    "mse_paired_estimate", "chi_additive", "chi_multiplicative")}})
    return {"status": "completed", "records": records, "summary_by_noise": groups,
            "limitations": [
                "Synthetic one-dimensional data; fixed filter width, no real-world generalization.",
                "The paired output is algebraically the median estimator, not a separate denoiser.",
                "Clean signal is used only for evaluation, not by deployable estimators.",
                "Xi is not factual correctness or a thermodynamic entropy measurement.",
                "The constant-shift test shows representation sensitivity; for logits a common shift preserves softmax probabilities.",
                "Both V3 formulas use illustrative parameters, not fitted or validated controllers.",
                "No physical phase conjugation, optical hardware, or quantum device is implemented."]}


def timed_gpu_scaling(output, device):
    rows = []
    for n in (1024, 16384, 262144, 1048576, 4194304):
        x = torch.randn(n, device=device)
        estimate = torch.randn_like(x)
        for _ in range(10):
            result = 0.5 * (x + (x - 2 * estimate))
        torch.cuda.synchronize()
        samples = []
        for _ in range(30):
            start = time.perf_counter()
            result = 0.5 * (x + (x - 2 * estimate))
            torch.cuda.synchronize()
            samples.append((time.perf_counter() - start) * 1000)
        rows.append({"n": n, "median_ms": float(np.median(samples)),
                     "p95_ms": float(np.percentile(samples, 95)), "samples_ms": samples})
    hist3d(output / "gpu_scaling_3d.png",
           [(f"N={row['n']:,}", row["samples_ms"]) for row in rows],
           "Synchronized GPU expression timings by vector size", "Elapsed time (ms)")
    return {"status": "completed", "rows": rows,
            "scope": "Synchronized PyTorch expression, eager execution; excludes estimator, data transfer and compilation. Not single-instruction or end-to-end inference latency."}


def architecture_simulation(output, device, seed, trials):
    """Finite-precision and phase/gain mismatch model, not an optical device model."""
    generator = torch.Generator(device=device).manual_seed(seed + 5000)
    records = []
    for trial in range(trials):
        clean, y = make_signal(generator, 1024, 0.75, "gaussian", device)
        estimate = smooth(y, 5, "median")
        n_hat = y - estimate
        scale = max(float(y.abs().max().item()), float(n_hat.abs().max().item()), 1e-7)
        for bits in (8, 12, 16):
            limit = (2 ** (bits - 1)) - 1
            q_y = torch.round(torch.clamp(y / scale, -1, 1) * limit) * (scale / limit)
            q_n = torch.round(torch.clamp(n_hat / scale, -1, 1) * limit) * (scale / limit)
            for phase in (0.0, 0.05, 0.2):
                for gain_error in (0.0, 0.05):
                    # Model a complex residual after imperfect estimated-noise subtraction.
                    real = q_y - q_n * (1.0 + gain_error) * math.cos(phase)
                    imag = -q_n * (1.0 + gain_error) * math.sin(phase)
                    mse = ((real-clean)**2 + imag**2).mean()
                    records.append({"trial": trial, "bits": bits, "phase_error_radians": phase,
                        "gain_error_fraction": gain_error, "input_mse": float(((y-clean)**2).mean().item()),
                        "ideal_median_mse": float(((estimate-clean)**2).mean().item()),
                        "simulated_output_mse": float(mse.item()), "quantization_scale": scale})
    groups = []
    for phase in (0.0, 0.05, 0.2):
        groups.append((f"phase={phase} rad, gain error=0",
                       [r["simulated_output_mse"] for r in records
                        if r["phase_error_radians"] == phase and r["gain_error_fraction"] == 0.0]))
    hist3d(output / "architecture_sensitivity_3d.png", groups,
           "Simulated cancellation MSE under phase mismatch (all bit depths)", "MSE")
    summary = []
    for bits in (8, 12, 16):
        for phase in (0.0, 0.05, 0.2):
            for gain in (0.0, 0.05):
                selected = [r for r in records if r["bits"] == bits
                            and r["phase_error_radians"] == phase and r["gain_error_fraction"] == gain]
                summary.append({"bits": bits, "phase_error_radians": phase,
                                "gain_error_fraction": gain, "n": len(selected),
                                "mean_input_mse": float(np.mean([r["input_mse"] for r in selected])),
                                "mean_ideal_median_mse": float(np.mean([r["ideal_median_mse"] for r in selected])),
                                "mean_simulated_output_mse": float(np.mean([r["simulated_output_mse"] for r in selected]))})
    return {"status": "completed", "records": records, "summary": summary,
            "model": "Digital uniform quantization followed by a phenomenological complex gain/phase mismatch in estimated-noise subtraction.",
            "limitations": ["No physical optical device, quantum process, circuit simulation, foundry model or fabricated IC.",
                "The imaginary residual is treated as error in this abstract model; actual detector response is not modeled.",
                "Full scale is chosen per observation from the available signal and estimated noise; calibration cost is excluded.",
                "No area, energy, transistor count, power, fabrication feasibility or system-level performance is measured.",
                "Phase and gain offsets are illustrative values, not measured hardware characteristics."]}


def kernel_comparison(output, device):
    def kernel(x, n):
        return 0.5 * (x + (x - 2 * n))
    x = torch.randn(1048576, device=device)
    noise = torch.randn_like(x)
    start = time.perf_counter()
    fused = torch.compile(kernel)
    compiled_result = fused(x, noise)
    torch.cuda.synchronize()
    setup_seconds = time.perf_counter() - start
    eager_result = kernel(x, noise)
    torch.cuda.synchronize()
    max_difference = float((compiled_result - eager_result).abs().max().item())
    if not torch.allclose(compiled_result, eager_result, rtol=1e-5, atol=1e-6):
        raise ValueError(f"Compiled output does not match eager output; max difference {max_difference}")
    for _ in range(10):
        kernel(x, noise)
        fused(x, noise)
    torch.cuda.synchronize()
    measurements = {"eager": [], "compiled": []}
    for repeat in range(40):
        # Alternate order to reduce systematic drift.
        order = ("eager", "compiled") if repeat % 2 == 0 else ("compiled", "eager")
        for name in order:
            t0 = time.perf_counter()
            (kernel if name == "eager" else fused)(x, noise)
            torch.cuda.synchronize()
            measurements[name].append((time.perf_counter() - t0) * 1000)
    hist3d(output / "kernel_comparison_3d.png", list(measurements.items()),
           "Eager versus torch.compile, paired synchronized GPU timings", "Elapsed time (ms)")
    medians = {k: float(np.median(v)) for k, v in measurements.items()}
    return {"status": "completed", "n": len(x), "measurements_ms": measurements,
            "median_ms": medians, "eager_over_compiled_median": medians["eager"] / medians["compiled"],
            "compile_and_first_run_seconds": setup_seconds, "max_abs_difference": max_difference,
            "limitations": ["One operation and dimension, PyTorch implementation, not a bespoke IC.",
                "Both implementations require an existing noise estimate; estimator and transfers excluded.",
                "GPU eager and compiler optimizations depend on version, warmup and hardware.",
                "Compiled mode is not guaranteed to use one hardware instruction; no energy measured.",
                "Speedup alone says nothing about accuracy, quantum advantage or general superiority."]}


def fetch_evidence(session, question):
    endpoint = "https://en.wikipedia.org/w/api.php"
    response = session.get(endpoint, params={"action": "query", "list": "search",
        "srsearch": question, "srlimit": 1, "format": "json"}, timeout=25)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise ValueError(f"Wikipedia search API error: {payload['error']}")
    found = payload["query"]["search"]
    if not found:
        return {"status": "no_results", "query": question, "title": None,
                "url": None, "extract": "", "extract_sha256": sha256("")}
    title = found[0]["title"]
    response = session.get(endpoint, params={"action": "query", "prop": "extracts|info",
        "titles": title, "exintro": 1, "explaintext": 1, "inprop": "url",
        "format": "json", "formatversion": 2}, timeout=25)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise ValueError(f"Wikipedia extract API error: {payload['error']}")
    page = payload["query"]["pages"][0]
    extract = page.get("extract", "")[:4000]
    if not extract:
        return {"status": "empty_extract", "query": question, "title": title,
                "url": page.get("fullurl"), "extract": "", "extract_sha256": sha256("")}
    return {"status": "ok", "query": question, "title": title, "url": page.get("fullurl"),
            "extract": extract, "extract_sha256": sha256(extract)}


def score_options(model, tokenizer, question, choices, evidence, device):
    prompt = ("Answer the question by choosing the best answer. "
              "Use the supplied evidence only if relevant.\n")
    if evidence is not None:
        prompt += f"Retrieved text (may be incomplete or wrong):\n{evidence}\n"
    prompt += f"Question: {question}\nAnswer:"
    prefix = tokenizer(prompt, add_special_tokens=True, return_tensors="pt")["input_ids"][0]
    scores = []
    with torch.inference_mode():
        for choice in choices:
            answer = tokenizer(" " + choice, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
            if len(answer) == 0:
                raise ValueError("An answer option tokenized to an empty sequence")
            seq = torch.cat((prefix, answer)).unsqueeze(0).to(device)
            logits = model(input_ids=seq).logits[0, len(prefix)-1:len(prefix)+len(answer)-1].float()
            log_probs = torch.log_softmax(logits, dim=-1)
            score = log_probs.gather(1, answer.to(device)[:, None]).sum().item()
            scores.append(float(score))
    probs = torch.softmax(torch.tensor(scores, dtype=torch.float64), dim=0).numpy()
    return {"scores_sum_logprob": scores, "choice_probabilities": probs.tolist(),
            "prediction": int(np.argmax(scores)), "confidence": float(np.max(probs)),
            "xi_option_scores": float(xi(torch.tensor(scores)).item()),
            "prompt_sha256": sha256(prompt)}


def auc_correct(scores, labels):
    positives = [s for s, label in zip(scores, labels) if label]
    negatives = [s for s, label in zip(scores, labels) if not label]
    if not positives or not negatives:
        return None
    return float(np.mean([1.0 if p > n else 0.5 if p == n else 0.0
                          for p in positives for n in negatives]))


def bootstrap_mean_interval(values, seed=108, repeats=2000):
    a = np.asarray(values, dtype=float)
    if not len(a):
        return None
    rng = np.random.default_rng(seed)
    means = [float(rng.choice(a, size=len(a), replace=True).mean()) for _ in range(repeats)]
    return [float(x) for x in np.quantile(means, [0.025, 0.975])]


def summarize_choices(rows, condition, thresholds):
    r = [x for x in rows if x["split"] == "test" and condition in x["conditions"]]
    if not r:
        return {"n": 0, "status": "no_available_results"}
    correct = [int(x["conditions"][condition]["correct"]) for x in r]
    confidence = [x["conditions"][condition]["confidence"] for x in r]
    xis = [x["conditions"][condition]["xi_option_scores"] for x in r]
    modes = {}
    for mode in ("confidence", "xi", "both"):
        selected = [x for x in r if
                    (mode == "xi" or x["conditions"][condition]["confidence"] >= thresholds["confidence"])
                    and (mode == "confidence" or x["conditions"][condition]["xi_option_scores"] >= thresholds["xi"])]
        modes[mode] = {"answered": len(selected), "coverage": len(selected) / len(r),
                       "selective_accuracy": (float(np.mean([x["conditions"][condition]["correct"]
                           for x in selected])) if selected else None),
                       "abstained": len(r) - len(selected)}
    return {"status": "completed", "n": len(r), "accuracy": float(np.mean(correct)),
            "accuracy_bootstrap_95_interval": bootstrap_mean_interval(correct),
            "thresholds": thresholds, "monitor_modes": modes,
            "confidence_correctness_auroc": auc_correct(confidence, correct),
            "xi_correctness_auroc": auc_correct(xis, correct),
            "mean_confidence": float(np.mean(confidence))}


def factual_study(output, args, device, checkpoint):
    print("Loading public TruthfulQA MC1 and model...", flush=True)
    ds = load_dataset(DATASET_ID, "multiple_choice", split="validation")
    rng = np.random.default_rng(args.seed)
    selected_indices = rng.permutation(len(ds))[:args.calibration + args.questions].tolist()
    if len(selected_indices) < args.calibration + args.questions:
        raise ValueError("Requested more calibration and test items than the dataset contains.")
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=torch.float16).to(device).eval()
    session = PacedSession()
    session.headers.update({"User-Agent": USER_AGENT})
    rows = []
    retrieval_failures = 0
    for pos, index in enumerate(selected_indices):
        example = ds[index]
        choices = list(example["mc1_targets"]["choices"])
        labels = list(example["mc1_targets"]["labels"])
        if len(choices) != len(labels) or sum(labels) != 1:
            raise ValueError(f"Unexpected MC1 labels at dataset index {index}")
        # Randomize answer order independently of model condition.
        order = rng.permutation(len(choices))
        choices = [choices[i] for i in order]
        correct = [labels[i] for i in order].index(1)
        row = {"dataset_index": index, "split": "calibration" if pos < args.calibration else "test",
               "question": example["question"], "choices": choices, "correct_index": correct,
               "conditions": {}}
        try:
            row["retrieval"] = fetch_evidence(session, row["question"])
        except (requests.RequestException, ValueError, KeyError, RuntimeError) as exc:
            retrieval_failures += 1
            row["retrieval"] = {"status": "error", "error": f"{type(exc).__name__}: {exc}"}
        for condition in ("internal", "retrieved"):
            if condition == "retrieved" and row["retrieval"]["status"] != "ok":
                continue
            evidence = None if condition == "internal" else row["retrieval"]["extract"]
            result = score_options(model, tokenizer, row["question"], choices, evidence, device)
            result["correct"] = result["prediction"] == correct
            row["conditions"][condition] = result
        rows.append(row)
        if (pos + 1) % 10 == 0 or pos + 1 == len(selected_indices):
            print(f"Factual questions: {pos+1}/{len(selected_indices)}, retrieval errors: {retrieval_failures}", flush=True)
            checkpoint["factual"] = {"status": "in_progress", "records": rows,
                                      "retrieval_errors": retrieval_failures}
            save_json(output / "sri_results.json", checkpoint)
    metadata = {"dataset": DATASET_ID, "dataset_config": "multiple_choice",
                "dataset_fingerprint": getattr(ds, "_fingerprint", None), "model": args.model,
                "model_revision": getattr(model.config, "_commit_hash", None),
                "tokenizer_revision": tokenizer.init_kwargs.get("_commit_hash")}
    return finalize_factual(output, rows, retrieval_failures, metadata)


def finalize_factual(output, rows, retrieval_failures, metadata):
    thresholds, summaries = {}, {}
    for condition in ("internal", "retrieved"):
        calibration = [x["conditions"][condition] for x in rows
                       if x["split"] == "calibration" and condition in x["conditions"]]
        if len(calibration) < 10:
            summaries[condition] = {"status": "insufficient_calibration", "n_calibration": len(calibration)}
            continue
        thresholds[condition] = {
            "confidence": float(np.quantile([r["confidence"] for r in calibration], 0.2)),
            "xi": float(np.quantile([r["xi_option_scores"] for r in calibration], 0.2))}
        summaries[condition] = summarize_choices(rows, condition, thresholds[condition])
    for x in rows:
        for condition, r in x["conditions"].items():
            if condition in thresholds:
                r["confidence_warning"] = r["confidence"] < thresholds[condition]["confidence"]
                r["xi_warning"] = r["xi_option_scores"] < thresholds[condition]["xi"]
                r["adaptive_decision"] = ("uncertain" if r["confidence_warning"] or r["xi_warning"]
                                          else "answer")
                r["monitor_reasons"] = []
                if r["confidence_warning"]:
                    r["monitor_reasons"].append("choice confidence below calibration threshold")
                if r["xi_warning"]:
                    r["monitor_reasons"].append("option-score Xi below calibration threshold")
            else:
                r["adaptive_decision"] = "assessment_unavailable"
                r["monitor_reasons"] = ["insufficient calibration data"]
            if condition == "retrieved":
                r["evidence_status"] = ("retrieved text supplied; relevance and factual support not verified")
            r["message_when_uncertain"] = ("The assessment is uncertain or unavailable; "
                "the available information does not establish a reliable answer. "
                "Check an authoritative source relevant to the question.")
    pairs = [x for x in rows if x["split"] == "test" and
             "internal" in x["conditions"] and "retrieved" in x["conditions"]]
    paired_summary = {"n": len(pairs)}
    if pairs:
        differences = [int(x["conditions"]["retrieved"]["correct"]) -
                       int(x["conditions"]["internal"]["correct"]) for x in pairs]
        paired_summary.update({
            "internal_accuracy": float(np.mean([x["conditions"]["internal"]["correct"] for x in pairs])),
            "retrieved_accuracy": float(np.mean([x["conditions"]["retrieved"]["correct"] for x in pairs])),
            "retrieved_minus_internal_accuracy": float(np.mean(differences)),
            "difference_bootstrap_95_interval": bootstrap_mean_interval(differences)})
    distributions = [(condition, [x["conditions"][condition]["xi_option_scores"]
                                  for x in rows if x["split"] == "test" and condition in x["conditions"]])
                     for condition in ("internal", "retrieved")]
    if any(vals for _, vals in distributions):
        hist3d(output / "factual_xi_3d.png", distributions,
               "Xi on answer-option log-likelihoods, held-out questions",
               "Xi (option scores; not truth or calibrated uncertainty)")
    factual_status = ("completed" if retrieval_failures == 0 and
                      all(v["status"] == "completed" and v["n"] > 0 for v in summaries.values())
                      else "partial_failure")
    return {"status": factual_status, **metadata,
            "retrieval_errors": retrieval_failures, "summaries": summaries, "paired_summary": paired_summary,
            "retrieval_status_counts": {status: sum(r["retrieval"]["status"] == status for r in rows)
                                        for status in ("ok", "error", "no_results", "empty_extract")},
            "thresholds": thresholds, "records": rows,
            "limitations": [
                "TruthfulQA MC1 is a multiple-choice proxy, not open-ended hallucination prevention.",
                "Wikipedia search retrieves one public article extract, not verified evidence or an entailment check.",
                "Retrieval failures and empty results are excluded from retrieved-condition metrics; inspect coverage.",
                "A completed run may contain legitimate no-results or empty-extract cases excluded from retrieved-condition metrics.",
                "Adaptive behavior is a comparison of heuristic confidence and Xi threshold-based abstention, not factual correction.",
                "Higher Xi is preselected as more reliable, not established; thresholds are not clinically or commercially validated.",
                "No source-specific support/conflict detection is implemented.",
                "Scores are sums of answer-token log probabilities, susceptible to answer length and prompt effects.",
                "Fixed calibration quantile is a heuristic, not a guarantee of calibration or coverage.",
                "Bootstrap intervals reflect sample variation only, not dataset bias, contamination or model uncertainty.",
                "The true answer label is used for evaluation only; no ground truth is included in prompts.",
                "This does not test physical cancellation or V3 damping in the LLM.",
                "Model and dataset contamination, retrieval changes and source errors are possible."]}


def resume_retrieval(source, output, device):
    raw = Path(source).read_bytes()
    original = json.loads(raw)
    if original.get("schema") != "sri-colab-research-v1":
        raise ValueError("Unsupported source report schema.")
    factual = original["factual"]
    rows = factual["records"]
    if not rows or any("internal" not in row["conditions"] for row in rows):
        raise ValueError("Source must contain complete internal-condition records.")
    revision = factual.get("model_revision")
    if not revision:
        raise ValueError("Cannot safely resume: original model revision was not recorded.")
    output.mkdir(parents=True, exist_ok=True)
    if (output / "sri_results.json").exists() or Path(source).resolve() == (output / "sri_results.json").resolve():
        raise ValueError("Resume requires a fresh output directory to preserve the original report.")
    report = original
    prior_resume = report.get("resume_provenance")
    if prior_resume:
        report.setdefault("resume_history", []).append(prior_resume)
    report["resume_provenance"] = {
        "original_json_sha256": hashlib.sha256(raw).hexdigest(),
        "resumed_utc": datetime.now(timezone.utc).isoformat(),
        "resume_script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "model_revision": revision,
        "note": "Original numerical and internal results retained; newly retrieved results use the same recorded model revision. Original tokenizer revision was not recorded. Original plot files are not copied; retain the original ZIP."}
    report["status"] = "in_progress"
    save_json(output / "sri_results.json", report)
    tokenizer = AutoTokenizer.from_pretrained(factual["model"], revision=revision)
    model = AutoModelForCausalLM.from_pretrained(
        factual["model"], revision=revision, torch_dtype=torch.float16).to(device).eval()
    session = PacedSession()
    session.headers.update({"User-Agent": USER_AGENT})
    for row in rows:
        if "retrieved" in row["conditions"] or row["retrieval"]["status"] != "error":
            continue
        old_retrieval = row["retrieval"]
        try:
            retrieved = fetch_evidence(session, row["question"])
            if retrieved["status"] == "ok":
                result = score_options(model, tokenizer, row["question"], row["choices"],
                                       retrieved["extract"], device)
                result["correct"] = result["prediction"] == row["correct_index"]
                row["conditions"]["retrieved"] = result
            row["retrieval"] = retrieved
            row["previous_retrieval"] = old_retrieval
            row.pop("resume_retrieval_error", None)
        except (requests.RequestException, ValueError, KeyError, RuntimeError) as exc:
            row["resume_retrieval_error"] = f"{type(exc).__name__}: {exc}"
            save_json(output / "sri_results.json", report)
            print(f"Retrieval stopped: {row['resume_retrieval_error']}", flush=True)
            break
        save_json(output / "sri_results.json", report)
        print(f"Retrieval resumed for dataset item {row['dataset_index']}: {row['retrieval']['status']}", flush=True)
    errors = sum(row["retrieval"]["status"] == "error" for row in rows)
    metadata = {k: factual.get(k) for k in ("dataset", "dataset_config", "dataset_fingerprint",
                                             "model", "model_revision", "tokenizer_revision")}
    report["factual"] = finalize_factual(output, rows, errors, metadata)
    report["factual"]["limitations"].append(
        "Retrieved-condition results may span separate runs and times; original tokenizer revision was not recorded.")
    report["failed_stages"] = [s for s in ("numerical", "scaling", "architecture_simulation",
                                            "kernel_comparison", "factual")
                                if report.get(s, {}).get("status") != "completed"]
    report["status"] = "partial_failure" if report["failed_stages"] else "completed"
    report["resume_provenance"]["completed_utc"] = datetime.now(timezone.utc).isoformat()
    report["resume_provenance"]["files"] = sorted(p.name for p in output.iterdir() if p.is_file())
    save_json(output / "sri_results.json", report)
    print(f"Resume status: {report['status']}; remaining retrieval errors: {errors}", flush=True)
    if report["failed_stages"]:
        raise SystemExit(1)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-dir", default="/content/sri_results")
    parser.add_argument("--model", default=MODEL_ID)
    parser.add_argument("--questions", type=int, default=100)
    parser.add_argument("--calibration", type=int, default=40)
    parser.add_argument("--trials", type=int, default=20)
    parser.add_argument("--seed", type=int, default=108)
    parser.add_argument("--resume-retrieval", help="Path to original sri_results.json; output must be a fresh directory.")
    args = parser.parse_args()
    if args.questions < 1 or args.calibration < 10 or args.trials < 1:
        parser.error("Use questions >= 1, calibration >= 10, and trials >= 1")
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required; select a Colab GPU runtime and rerun.")
    device = torch.device("cuda")
    if "A100" not in torch.cuda.get_device_name(0).upper():
        print("WARNING: GPU is not an A100; recording actual hardware.", flush=True)
    if args.resume_retrieval:
        resume_retrieval(args.resume_retrieval, output, device)
        return
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    script_path = Path(__file__)
    report = {"schema": "sri-colab-research-v1", "started_utc": datetime.now(timezone.utc).isoformat(),
              "status": "in_progress", "configuration": vars(args),
              "script_sha256": hashlib.sha256(script_path.read_bytes()).hexdigest(),
              "environment": {"python": sys.version, "platform": platform.platform(),
                              "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
                              "gpu": torch.cuda.get_device_name(0),
                              "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
                              "numpy": np.__version__, "matplotlib": matplotlib.__version__,
                              "requests": requests.__version__},
              "interpretation": ["Classical GPU experiments, not quantum advantage or optical hardware.",
                                 "Timings do not prove O(1), and Xi is not a truth or entropy measure.",
                                 "Reported results must not be used to claim perfect error correction."],
              "artifacts": ["sri_results.json", "xi_before_after_3d.png",
                            "residuals_3d.png", "v3_damping_3d.png",
                            "gpu_scaling_3d.png", "architecture_sensitivity_3d.png",
                            "kernel_comparison_3d.png", "factual_xi_3d.png"]}
    try:
        result = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                                capture_output=True, text=True, check=True)
        report["environment"]["driver"] = result.stdout.strip()
    except (OSError, subprocess.SubprocessError) as exc:
        report["environment"]["driver_error"] = f"{type(exc).__name__}: {exc}"
    save_json(output / "sri_results.json", report)
    failures = []
    for name in ("numerical", "scaling", "architecture_simulation", "kernel_comparison", "factual"):
        print(f"Starting {name}...", flush=True)
        try:
            if name == "numerical":
                report[name] = numerical_study(output, device, args.seed, args.trials)
            elif name == "scaling":
                report[name] = timed_gpu_scaling(output, device)
            elif name == "architecture_simulation":
                report[name] = architecture_simulation(output, device, args.seed, args.trials)
            elif name == "kernel_comparison":
                report[name] = kernel_comparison(output, device)
            else:
                report[name] = factual_study(output, args, device, report)
            if report[name]["status"] != "completed":
                failures.append(name)
                print(f"ERROR: {name} returned status {report[name]['status']}", flush=True)
        except Exception as exc:
            # Keep completed independent stages and checkpoints, but mark the run failed.
            previous = report.get(name, {})
            report[name] = {**previous, "status": "failed",
                            "error": f"{type(exc).__name__}: {exc}",
                            "traceback": traceback.format_exc()}
            failures.append(name)
            print(f"ERROR in {name}: {type(exc).__name__}: {exc}", flush=True)
            traceback.print_exc()
        save_json(output / "sri_results.json", report)
    report["status"] = "partial_failure" if failures else "completed"
    report["failed_stages"] = failures
    report["completed_utc"] = datetime.now(timezone.utc).isoformat()
    report["artifacts_present"] = sorted(p.name for p in output.iterdir() if p.is_file())
    save_json(output / "sri_results.json", report)
    print(f"Finished with status {report['status']}; artifacts: {output}", flush=True)
    if failures:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


## Run experiments

Progress prints appear below. Partial results are checkpointed in `sri_results.json` every ten factual questions. The run deliberately reports a failed stage rather than silently treating missing data or failed retrieval as a successful comparison. If a stage fails, continue to the packaging cell to save the available results and error information.


In [ ]:
import subprocess, sys, shutil
shutil.copy2("/content/sri_colab_experiments.py", run_dir / "sri_colab_experiments.py")
cmd = [sys.executable, "-u", "/content/sri_colab_experiments.py",
       "--output-dir", str(run_dir), "--model", MODEL,
       "--calibration", str(N_CALIBRATION), "--questions", str(N_TEST),
       "--trials", str(N_TRIALS), "--seed", str(SEED)]
if RESUME_RETRIEVAL:
    cmd.extend(["--resume-retrieval", str(source_report)])
print("Running:", " ".join(cmd), flush=True)
result = subprocess.run(cmd)
print("Exit code:", result.returncode, "— 0 means all stages completed.")
if result.returncode:
    print("A stage failed or was incomplete. Preserve the JSON and inspect its error and status fields.")


## Inspect and package artifacts

Share the ZIP, including the JSON, PNGs and script. The JSON includes per-question records and retrieved public excerpts. It records test scope and limitations; keep those with any VC or patent-attorney summary. A result is not independently verified merely because it appears in the report.


In [ ]:
import json, shutil
result_path = run_dir / "sri_results.json"
if result_path.exists():
    result_data = json.loads(result_path.read_text())
    print("Overall status:", result_data.get("status"))
    for section in ("numerical", "scaling", "architecture_simulation", "kernel_comparison", "factual"):
        print(section, ":", result_data.get(section, {}).get("status", "not started"))
    print("Factual summary:", json.dumps(result_data.get("factual", {}).get("summaries", {}), indent=2))
else:
    print("No JSON generated; inspect prior cell output for the startup failure.")
print("Files:", [p.name for p in run_dir.iterdir()])
from IPython.display import display, Image
for image in sorted(run_dir.glob("*_3d.png")):
    print(image.name)
    display(Image(filename=str(image)))
zip_path = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("ZIP:", zip_path)
if not SAVE_TO_DRIVE:
    from google.colab import files
    files.download(zip_path)
